In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
import warnings
import mlflow
warnings.filterwarnings('ignore')

sys.path.append('../scripts')
from data_pipeline import load_and_prepare_data, load_single_channel
from feature_engineering import build_model_input
from algorithms import gpr
import plots, evaluate

# Set MLflow tracking to project root (not notebooks folder)
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("GPR_Battery_Degradation")

In [ ]:
# Fix the data path - using Li_highC_25C which exists
X_train_loso, X_test_loso, y_train_loso, y_test_loso = load_and_prepare_data(data_folder="../data/Li_highC_25C")

In [ ]:
cap_model = gpr.train_capacity_gpr_fast(
    X_train_loso, y_train_loso,
    subset_size=300,
    top_k_freqs=None  
)

In [ ]:
# Start MLflow run for GPR experiment
if mlflow.active_run():
    mlflow.end_run()

mlflow.start_run(run_name="GPR_LOSO_04-03-24")

# Log parameters
mlflow.log_param("model_type", "Gaussian Process Regression")
mlflow.log_param("split_method", "LOSO")
mlflow.log_param("data_folder", "04-03-24")
mlflow.log_param("subset_size", 300)
mlflow.log_param("train_size", X_train_loso.shape[0])
mlflow.log_param("test_size", X_test_loso.shape[0])
mlflow.log_param("n_features", X_train_loso.shape[1])

In [ ]:
y_pred_mean, y_pred_std = gpr.predict_fast(cap_model, X_test_loso)

In [ ]:
rmse, r2, mse, mae = evaluate.evaluate_model(y_test_loso, y_pred_mean)
print(f"RMSE: {rmse:.4f}\nR²: {r2:.4f}\nMSE: {mse:.4f}\nMAE: {mae:.4f}")
plots.model_predictions(y_test_loso, y_pred_mean, y_pred_std, title_prefix="GPR (fast), LOSO")

In [ ]:
X_train_bin, X_test_bin, y_train_bin, y_test_bin = load_and_prepare_data(data_folder="../data/Li_highC_25C",method='bin_and_split')

In [ ]:
cap_model = gpr.train_capacity_gpr_fast(
    X_train_bin, y_train_bin,
    top_k_freqs=None  
)

In [ ]:
y_pred_mean, y_pred_std = gpr.predict_fast(cap_model, X_test_bin)

In [ ]:
rmse, r2, mse, mae = evaluate.evaluate_model(y_test_bin, y_pred_mean)
print(f"RMSE: {rmse:.4f}\nR²: {r2:.4f}\nMSE: {mse:.4f}\nMAE: {mae:.4f}")
plots.model_predictions(y_test_bin, y_pred_mean, y_pred_std, title_prefix="GPR (fast), Binning")

In [ ]:
w = gpr.ard_frequency_weights(cap_model)       # shape (66,)
plots.gpr_weights(w)

In [ ]:
# Split into Re/Im and aggregate per frequency
n_freq = w.size // 2
w_re, w_im = w[:n_freq], w[n_freq:]
w_mean = (w_re + w_im) / 2

# Top frequencies
top = np.argsort(w_mean)[::-1][:10]
print("Top-10 freqs (index):", top)
print("Weights:", w_mean[top])

In [ ]:
def loso(train_channels, test_channels):
    train_channels_data = {}
    test_channels_data = {}
    for channel in train_channels: train_channels_data[channel] = load_single_channel("../data/Li_highC_25C", channel)    
    for channel in test_channels: test_channels_data[channel] = load_single_channel("../data/Li_highC_25C", channel)
    train_dfs = list(train_channels_data.values())
    test_dfs = list(test_channels_data.values())

    return pd.concat(train_dfs, ignore_index=True), pd.concat(test_dfs, ignore_index=True)


In [ ]:
CHANNELS = ['A1','A2','A3','A4','A5','A6','A7','A8']
results = []

for test_cell in CHANNELS:
    train_cells = [c for c in CHANNELS if c != test_cell]
    df_train, df_test = loso(train_cells, [test_cell])

    X_train, y_train = build_model_input(df_train)
    X_test,  y_test  = build_model_input(df_test)

    model = gpr.train_capacity_gpr_fast(X_train, y_train, subset_size=500)
    y_pred, y_std = gpr.predict_fast(model, X_test)

    rmse, r2, mse, mae = evaluate.evaluate_model(y_test, y_pred)
    results.append((test_cell, rmse, mae, r2))


In [ ]:
results

# Plot weights

In [ ]:
# Process and save ARD weights (same as before)
W_re = np.vstack(W_re)
W_im = np.vstack(W_im)

re_mean, re_std = W_re.mean(axis=0), W_re.std(axis=0)
im_mean, im_std = W_im.mean(axis=0), W_im.std(axis=0)
avg_mean = (re_mean + im_mean) / 2.0
avg_std = np.sqrt((re_std**2 + im_std**2) / 2.0)

freqs_hz = [
    0.999, 1.33, 1.78, 2.37, 3.16, 4.22, 5.62, 7.5, 
    10.0, 13.3, 17.8, 23.7, 31.6, 42.2, 56.2, 75.0, 
    102.0, 135.0, 178.0, 237.0, 316.0, 422.0, 564.0, 750.0, 
    1000.0, 1330.0, 1780.0, 2370.0, 3160.0, 4220.0, 5620.0, 7500.0, 
    10000.0
]

# Save ARD weights
ard_df = pd.DataFrame({
    "freq_hz": freqs_hz,
    "w_re_mean": re_mean, "w_re_std": re_std,
    "w_im_mean": im_mean, "w_im_std": im_std,
    "w_avg_mean": avg_mean, "w_avg_std": avg_std,
})
ard_csv_path = save_dir / "ard_weights_across_folds.csv"
ard_df.to_csv(ard_csv_path, index=False)

print(f"\nARD weights saved to: {ard_csv_path}")
print(f"All 8 models saved to: {model_dir}")



In [ ]:
import os

plt = plots.ard_summary(
    freqs_hz, re_mean, re_std, im_mean, im_std, avg_mean
)

png_path = os.path.join(save_dir, "ard_weights_across_folds.png")

plt.savefig(png_path, dpi=180)

print(f"Saved ARD summary to:\n- {csv_path}\n- {png_path}")

# Cross fold LOSO with highest frequencies

In [ ]:
CHANNELS = ['A1','A2','A3','A4','A5','A6','A7','A8']
results = []
freqs = [0.999,31.600,2370.000,5620.000,7500.000]

for test_cell in CHANNELS:
    train_cells = [c for c in CHANNELS if c != test_cell]
    df_train, df_test = loso(train_cells, [test_cell])

    X_train, y_train = build_model_input(df_train, freqs=freqs)
    X_test,  y_test  = build_model_input(df_test, freqs=freqs)

    model = gpr.train_capacity_gpr_fast(X_train, y_train, subset_size=500)
    y_pred, y_std = gpr.predict_fast(model, X_test)

    rmse, r2, mse, mae = evaluate.evaluate_model(y_test, y_pred)
    results.append((test_cell, rmse, mae, r2))


In [ ]:
import os, json

df = pd.DataFrame(results, columns=["cell", "rmse", "mae", "r2"]).sort_values("cell")

summary = pd.DataFrame({
    "cell": ["_mean_"],
    "rmse": [df["rmse"].mean()],
    "mae":  [df["mae"].mean()],
    "r2":   [df["r2"].mean()],
})
df_out = pd.concat([df, summary], ignore_index=True)

save_dir = os.path.join("../results", "gpr")
os.makedirs(save_dir, exist_ok=True)

csv_path  = os.path.join(save_dir, "gpr-select-freqs_8_fold_cv_results.csv")
json_path = os.path.join(save_dir, "gpr-select-freqs_8_fold_cv_results.json")

df_out.to_csv(csv_path, index=False)
with open(json_path, "w") as f:
    json.dump(df.to_dict(orient="records"), f, indent=2)

print(f"Results saved to:\n  • {csv_path}\n  • {json_path}")
print("\n Fold-wise summary:\n", df_out)


In [ ]:
from pathlib import Path

# === Paths ===
results_dir = Path("../results/gpr")
full_file = results_dir / "gpr_8_fold_cv_results.csv"
top_file  = results_dir / "gpr-select-freqs_8_fold_cv_results.csv"

# === Load
df_full = pd.read_csv(full_file)
df_top  = pd.read_csv(top_file)

# Normalize column name: use 'channel'
name_col = "channel" if "channel" in df_full.columns else "cell"
df_full = df_full.rename(columns={name_col: "channel"})
name_col = "channel" if "channel" in df_top.columns else "cell"
df_top  = df_top.rename(columns={name_col: "channel"})

# Drop optional summary row (e.g., "_mean_") if present
df_full = df_full[df_full["channel"].astype(str) != "_mean_"].copy()
df_top  = df_top[df_top["channel"].astype(str)  != "_mean_"].copy()

# Tag model type
df_full["model"] = "Full Spectrum"
df_top["model"]  = "Top-5 Frequencies"

# Combine + sort channels
df = pd.concat([df_full, df_top], ignore_index=True)
order = ["A1","A2","A3","A4","A5","A6","A7","A8"]
df["channel"] = pd.Categorical(df["channel"], categories=order, ordered=True)
df = df.sort_values("channel")

# --- R² comparison plot ---
fig, ax = plt.subplots()
for model, sub in df.groupby("model"):
    ax.plot(sub["channel"], sub["r2"], marker="o", label=model)

ax.set_xlabel("Test Channel")
ax.set_ylabel(r"$R^2$ Score")
ax.set_title("8-Fold LOSO GPR Performance")
ax.legend()

out_path = results_dir / "gpr_8fold_r2_comparison.png"
fig.savefig(out_path)
print(f"Saved figure to {out_path}")

plt.show()
